In [21]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [22]:
import sys

class TrackedObject:
    def __init__(self, **kwargs):
        # history инициализируем через super().__setattr__, чтобы не попасть в лог
        super().__setattr__('history', [])
        # Записываем все переданные kwargs как атрибуты (через наш __setattr__, чтобы залогировать)
        for name, value in kwargs.items():
            setattr(self, name, value)

    def __setattr__(self, name, value):
        # Размер __dict__ ДО операции
        # Используем object.__getattribute__, чтобы не вызвать рекурсию
        current_dict = object.__getattribute__(self, '__dict__')
        size_before = sys.getsizeof(current_dict)
        keys_before = set(current_dict.keys())
        existed = name in current_dict
        old_value = current_dict.get(name, '<absent>')

        # Выполняем реальную установку атрибута
        super().__setattr__(name, value)

        # Размер __dict__ ПОСЛЕ операции
        size_after = sys.getsizeof(current_dict)  # тот же объект dict, уже обновлён

        action = 'updated' if existed else 'added'
        event = (
            f"SET   '{name}': {old_value!r} -> {value!r} "
            f"[action={action}, "
            f"__dict__ size: {size_before}B -> {size_after}B, "
            f"keys: {sorted(keys_before)} -> {sorted(current_dict.keys())}]"
        )
        # Добавляем запись в history через super(), чтобы не вызвать рекурсию
        object.__getattribute__(self, 'history').append(event)

    def __delattr__(self, name):
        current_dict = object.__getattribute__(self, '__dict__')
        size_before = sys.getsizeof(current_dict)
        keys_before = set(current_dict.keys())
        old_value = current_dict.get(name, '<absent>')

        try:
            # Выполняем реальное удаление атрибута
            super().__delattr__(name)
        except AttributeError as exc:
            # Атрибут не существует — логируем ошибку и пробрасываем исключение
            event = (
                f"DEL   '{name}': ERROR — {exc} "
                f"[__dict__ size: {size_before}B (unchanged), "
                f"keys: {sorted(keys_before)}]"
            )
            object.__getattribute__(self, 'history').append(event)
            raise

        size_after = sys.getsizeof(current_dict)

        event = (
            f"DEL   '{name}': {old_value!r} "
            f"[__dict__ size: {size_before}B -> {size_after}B, "
            f"keys: {sorted(keys_before)} -> {sorted(current_dict.keys())}]"
        )
        object.__getattribute__(self, 'history').append(event)


obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x
try:
    del obj.q
except AttributeError:
    pass

print('=== obj.__dict__ ===')
print(obj.__dict__)
print()
print('=== history ===')
for event in obj.history:
    print(event)

=== obj.__dict__ ===
{'history': ["SET   'x': '<absent>' -> 1 [action=added, __dict__ size: 296B -> 296B, keys: ['history'] -> ['history', 'x']]", "SET   'y': '<absent>' -> 2 [action=added, __dict__ size: 296B -> 296B, keys: ['history', 'x'] -> ['history', 'x', 'y']]", "SET   'z': '<absent>' -> 3 [action=added, __dict__ size: 296B -> 296B, keys: ['history', 'x', 'y'] -> ['history', 'x', 'y', 'z']]", "SET   '__str__': '<absent>' -> <function <lambda> at 0x10c710300> [action=added, __dict__ size: 296B -> 296B, keys: ['history', 'x', 'y', 'z'] -> ['__str__', 'history', 'x', 'y', 'z']]", "DEL   'x': 1 [__dict__ size: 296B -> 296B, keys: ['__str__', 'history', 'x', 'y', 'z'] -> ['__str__', 'history', 'y', 'z']]", "DEL   'q': ERROR — 'TrackedObject' object has no attribute 'q' [__dict__ size: 296B (unchanged), keys: ['__str__', 'history', 'y', 'z']]"], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x10c710300>}

=== history ===
SET   'x': '<absent>' -> 1 [action=added, __dict__ size: 296B

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [23]:
# ---------------------------------------------------------------------------
# C3-linearization algorithm
# ---------------------------------------------------------------------------

def merge(seqs: list[list]) -> list:
    """
    Merge step of the C3 algorithm.

    Accepts a list of sequences (each is a list of classes).
    On every iteration:
      1. Take the 'head' (first element) of the first non-empty sequence.
      2. Check that this head does NOT appear in the 'tail' (seq[1:])
         of any other sequence — this is called a 'good head'.
      3. If a good head is found — append it to the result and remove
         it from all sequences.
      4. If no good head exists — the inheritance graph is inconsistent
         (MRO cannot be built), raise TypeError.
    """
    result = []
    seqs = [list(s) for s in seqs]   # work with copies

    while True:
        seqs = [s for s in seqs if s]  # drop empty sequences
        if not seqs:
            return result

        # Collect all 'tails' (everything except the first element)
        tails = {cls for s in seqs for cls in s[1:]}

        # Find a 'good head' — a head that is not in any tail
        good_head = None
        for seq in seqs:
            candidate = seq[0]
            if candidate not in tails:
                good_head = candidate
                break

        if good_head is None:
            remaining = [s[0] for s in seqs]
            raise TypeError(
                f'Cannot create a consistent MRO. '
                f'Conflicting heads: {remaining}'
            )

        result.append(good_head)
        # Remove the chosen head from every sequence
        for seq in seqs:
            if seq[0] is good_head:
                seq.pop(0)


def c3_linearize(cls) -> list:
    """
    Compute the C3 linearization of cls without using CPython internals.

    Formula:
        L[C] = [C] + merge(L[B1], L[B2], ..., L[Bn], [B1, B2, ..., Bn])

    where B1..Bn are the direct base classes of C.
    """
    bases = cls.__bases__

    # Base case: object (no bases)
    if not bases:
        return [cls]

    # Recursively linearize each base
    linearizations = [c3_linearize(b) for b in bases]
    # Append the list of bases itself as the last sequence
    linearizations.append(list(bases))

    return [cls] + merge(linearizations)


# ---------------------------------------------------------------------------
# Class hierarchy: two diamonds + shared ancestors
#
# Diamond 1: G -> C -> A и G -> D -> A
# Diamond 2: G -> D -> B и G -> E -> B
# G(C, D, E) captures both diamonds.
# Total: 7 user-defined classes (A, B, C, D, E, F, G) + object.
#
# Name conflict: method greet() is defined in both C and E.
# ---------------------------------------------------------------------------

class A:
    def who(self): return 'A'

class B:
    def who(self): return 'B'

class C(A):
    def who(self): return 'C'
    # conflicting method: also defined in E
    def greet(self): return 'Hello from C'

class D(A, B):
    """D inherits both A and B — first diamond."""
    def who(self): return 'D'

class E(B):
    def who(self): return 'E'
    # conflicting method: also defined in C
    def greet(self): return 'Hello from E'

class F(B):
    def who(self): return 'F'

class G(C, D, E):
    """
    G(C, D, E):
      C and D both pull in A  -> diamond 1
      D and E both pull in B  -> diamond 2
    """
    pass


# ---------------------------------------------------------------------------
# Pretty-print helper
# ---------------------------------------------------------------------------

def show_mro(cls):
    our     = [c.__name__ for c in c3_linearize(cls)]
    cpython = [c.__name__ for c in cls.__mro__]
    status  = 'OK  matches' if our == cpython else 'MISMATCH!'
    print(f'\n{"─" * 60}')
    print(f'Class        : {cls.__name__}')
    print(f'c3_linearize : {our}')
    print(f'__mro__      : {cpython}')
    print(f'Check        : {status}')


for cls in (A, B, C, D, E, F, G):
    show_mro(cls)


# ---------------------------------------------------------------------------
# Demonstrate name-conflict resolution via MRO
# ---------------------------------------------------------------------------

print('\n' + '=' * 60)
print('Name conflict: greet() is defined in both C and E')
print('=' * 60)

g = G()
print(f'g.greet() -> {g.greet()!r}')
print(f'g.who()   -> {g.who()!r}')
print()
print('MRO of G:', [c.__name__ for c in G.__mro__])
print('C comes before E in MRO -> C.greet() wins.')



────────────────────────────────────────────────────────────
Class        : A
c3_linearize : ['A', 'object']
__mro__      : ['A', 'object']
Check        : OK  matches

────────────────────────────────────────────────────────────
Class        : B
c3_linearize : ['B', 'object']
__mro__      : ['B', 'object']
Check        : OK  matches

────────────────────────────────────────────────────────────
Class        : C
c3_linearize : ['C', 'A', 'object']
__mro__      : ['C', 'A', 'object']
Check        : OK  matches

────────────────────────────────────────────────────────────
Class        : D
c3_linearize : ['D', 'A', 'B', 'object']
__mro__      : ['D', 'A', 'B', 'object']
Check        : OK  matches

────────────────────────────────────────────────────────────
Class        : E
c3_linearize : ['E', 'B', 'object']
__mro__      : ['E', 'B', 'object']
Check        : OK  matches

────────────────────────────────────────────────────────────
Class        : F
c3_linearize : ['F', 'B', 'object']
__mro

# Why this order of method resolution
## 1. LOCAL PRECEDENCE ORDER
   If a class is declared as C(B1, B2, ...), then in MRO(C). B1 always precedes B2, B2 precedes B3, etc. merge() never picks a candidate that appears in the 'tail' of another sequence, thus preserving the programmer-specified order. Example: G(C, D, E) -> MRO of G has C before D before E.

## 2. MONOTONICITY
   If X precedes Y in MRO(C), then X precedes Y in the MRO of every subclass of C. Adding a new subclass never reorders an already established precedence.

## 3. NO INCONSISTENCY
   If the inheritance graph is contradictory (e.g. A(B, C) and D(C, B) — the order of B and C is inverted). And in some class F (A, B), merge() will find no 'good head' and raise TypeError. C3 explicitly forbids ambiguous hierarchies.

## 4. RESOLVING THE greet() CONFLICT IN G
   G(C, D, E): C is listed first -> C.greet() beats E.greet(). This is a direct consequence of local precedence: by writing G(C, D, E) the programmer declares that C has priority over E.

## 5. DIAMONDS AND A SINGLE object
   Even though A and B appear in multiple branches, each class appears exactly once in the final MRO. merge() removes a class from all sequences as soon as it is chosen — no duplicates.

## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [24]:
class SecureBase:
    """
    Demonstrates CPython name mangling.

    Any identifier of the form __name (two leading underscores, at most one
    trailing underscore) defined inside a class body is textually replaced by
    _ClassName__name at compile time.  This happens BEFORE the bytecode is
    generated, so it is purely a syntactic transformation — not a runtime
    access-control mechanism.
    """

    def __init__(self):
        # Stored as '_SecureBase__secret_value' in __dict__
        self.__secret_value = 'base-secret'
        # Single leading underscore — convention only, NO mangling
        self._semi_private   = 'base-semi'
        # Plain public attribute
        self.public          = 'base-public'

    def get_secret(self):
        """Access via the mangled name inside the defining class — works fine."""
        return self.__secret_value


class SecureChild(SecureBase):
    """
    Inherits SecureBase and overrides both private-looking attributes.

    Key insight: self.__secret_value here is mangled to
    '_SecureChild__secret_value', which is a DIFFERENT key in __dict__ from
    '_SecureBase__secret_value'.  Both coexist without collision.
    """

    def __init__(self):
        super().__init__()                    # populates _SecureBase__secret_value
        # Creates a NEW key '_SecureChild__secret_value' — does NOT overwrite parent's
        self.__secret_value = 'child-secret'
        # Overwrites the same '_semi_private' key (no mangling)
        self._semi_private  = 'child-semi'
        self.public         = 'child-public'

    def dump_dict(self):
        """Return the raw instance __dict__ so we can inspect real key names."""
        return self.__dict__

    def get_own_secret(self):
        """Accesses '_SecureChild__secret_value' (mangled inside this class)."""
        return self.__secret_value


# ── Instantiate both classes ──────────────────────────────────────────────────
base  = SecureBase()
child = SecureChild()

SEP = '─' * 60

# ── 1. __dict__ — real storage keys ──────────────────────────────────────────
print(SEP)
print('1. __dict__ of SecureBase instance')
print(SEP)
for k, v in base.__dict__.items():
    print(f'  {k!r:40s} -> {v!r}')

print()
print(SEP)
print('2. __dict__ of SecureChild instance  (via dump_dict)')
print(SEP)
for k, v in child.dump_dict().items():
    print(f'  {k!r:40s} -> {v!r}')

# ── 2. dir() — filtered to show only relevant names ──────────────────────────
def mangled_names(obj):
    """Return only the mangled / semi-private names visible in dir()."""
    return [n for n in dir(obj) if n.startswith('_') and not n.startswith('__')]

print()
print(SEP)
print('3. dir() — single-underscore names for SecureBase instance')
print(SEP)
print(' ', mangled_names(base))

print()
print(SEP)
print('4. dir() — single-underscore names for SecureChild instance')
print(SEP)
print(' ', mangled_names(child))

# ── 3. Access via mangled names from outside the class ───────────────────────
print()
print(SEP)
print('5. Access mangled from outside:')
print(SEP)

# _SecureBase__secret_value is accessible from anywhere using the mangled name
print(f'  base._SecureBase__secret_value  = {base._SecureBase__secret_value!r}')
print(f'  child._SecureBase__secret_value = {child._SecureBase__secret_value!r}')
print(f'  child._SecureChild__secret_value= {child._SecureChild__secret_value!r}')

# Verify that the parent's get_secret() still reads the BASE slot, not the child's
print()
print(f'  child.get_secret()     (SecureBase method) -> {child.get_secret()!r}')
print(f'  child.get_own_secret() (SecureChild method)-> {child.get_own_secret()!r}')

# ── 4. Attempt direct access without mangling — AttributeError ───────────────
print()
print(SEP)
print('6. Direct access without mangling raises AttributeError:')
print(SEP)
try:
    _ = base.__secret_value
except AttributeError as exc:
    print(f'  base.__secret_value -> AttributeError: {exc}')

# ── 5. Summary table ─────────────────────────────────────────────────────────
print()
print(SEP)
print('7. Summary: real key names in __dict__')
print(SEP)
rows = [
    ('Attribute written', 'Stored key in __dict__',              'Mangled?'),
    ('self.__secret_value (in SecureBase)',  '_SecureBase__secret_value',  'YES'),
    ('self.__secret_value (in SecureChild)', '_SecureChild__secret_value', 'YES'),
    ('self._semi_private',                   '_semi_private',              'NO'),
    ('self.public',                          'public',                     'NO'),
]
col_w = [42, 30, 8]
for row in rows:
    print('  ' + '  '.join(str(c).ljust(w) for c, w in zip(row, col_w)))

────────────────────────────────────────────────────────────
1. __dict__ of SecureBase instance
────────────────────────────────────────────────────────────
  '_SecureBase__secret_value'              -> 'base-secret'
  '_semi_private'                          -> 'base-semi'
  'public'                                 -> 'base-public'

────────────────────────────────────────────────────────────
2. __dict__ of SecureChild instance  (via dump_dict)
────────────────────────────────────────────────────────────
  '_SecureBase__secret_value'              -> 'base-secret'
  '_semi_private'                          -> 'child-semi'
  'public'                                 -> 'child-public'
  '_SecureChild__secret_value'             -> 'child-secret'

────────────────────────────────────────────────────────────
3. dir() — single-underscore names for SecureBase instance
────────────────────────────────────────────────────────────
  ['_SecureBase__secret_value', '_semi_private']

────────────────

## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [25]:
class NoSlots:
    """Regular class — no __slots__.

    CPython allocates a per-instance __dict__ (a plain Python dict) to store
    all instance attributes.  A __weakref__ slot is also created automatically,
    so weak references work out of the box.
    """
    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlots:
    """Class with __slots__ = ('x', 'y').

    CPython replaces the per-instance __dict__ with a fixed array of C-level
    slot descriptors.  No __dict__ is created, no __weakref__ slot either —
    so both dynamic attribute assignment and weak references are impossible.
    """
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlotsWeak:
    """Class with __slots__ = ('x', '__weakref__').

    Explicitly adding '__weakref__' to __slots__ re-enables weak-reference
    support while still suppressing __dict__.  Dynamic attribute assignment
    remains impossible (only 'x' is a valid slot).
    """
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x


# ── Helper: describe a single instance ───────────────────────────────────────

def describe_instance(obj):
    """Return a dict with key metrics for *obj*."""
    has_dict    = hasattr(obj, "__dict__")
    has_weakref = hasattr(obj, "__weakref__")
    size_obj    = sys.getsizeof(obj)
    size_dict   = sys.getsizeof(obj.__dict__) if has_dict else None
    total_size  = size_obj + (size_dict or 0)
    dict_keys   = list(obj.__dict__.keys()) if has_dict else None

    slots = getattr(type(obj), "__slots__", None)

    return {
        "type"       : type(obj).__name__,
        "has_dict"   : has_dict,
        "has_weakref": has_weakref,
        "__slots__"  : slots,
        "size_obj"   : size_obj,
        "size_dict"  : size_dict,
        "total_size" : total_size,
        "dict_keys"  : dict_keys,
    }


# ── Instantiate series of objects ────────────────────────────────────────────

N = 5
objs_no = [NoSlots(i, i + 1)    for i in range(N)]
objs_ws = [WithSlots(i, i + 1)  for i in range(N)]
objs_ww = [WithSlotsWeak(i)      for i in range(N)]

# Use the first element for per-instance structural inspection
n  = objs_no[0]
w  = objs_ws[0]
ww = objs_ww[0]

SEP = "─" * 60

# ── 1. Structural comparison ──────────────────────────────────────────────────

print(SEP)
print("1. Structural comparison (hasattr / __slots__ / sizes)")
print(SEP)

header = f"{'Metric':<22} {'NoSlots':>12} {'WithSlots':>12} {'WithSlotsWeak':>14}"
print(header)
print("-" * len(header))

dn  = describe_instance(n)
dw  = describe_instance(w)
dww = describe_instance(ww)

rows = [
    ("has __dict__",    dn["has_dict"],    dw["has_dict"],    dww["has_dict"]),
    ("has __weakref__", dn["has_weakref"], dw["has_weakref"], dww["has_weakref"]),
    ("__slots__",       dn["__slots__"],   dw["__slots__"],   dww["__slots__"]),
    ("size_obj (B)",    dn["size_obj"],    dw["size_obj"],    dww["size_obj"]),
    ("size_dict (B)",   dn["size_dict"],   dw["size_dict"],   dww["size_dict"]),
    ("total_size (B)",  dn["total_size"],  dw["total_size"],  dww["total_size"]),
    ("dict_keys",       dn["dict_keys"],   dw["dict_keys"],   dww["dict_keys"]),
]
for metric, v1, v2, v3 in rows:
    print(f"  {metric:<20} {str(v1):>12} {str(v2):>12} {str(v3):>14}")

# ── 1b. Verify sizes are uniform across the whole series ─────────────────────

print()
print("  Size consistency across all", N, "instances in each series:")
for label, series in (("NoSlots", objs_no), ("WithSlots", objs_ws), ("WithSlotsWeak", objs_ww)):
    sizes = [sys.getsizeof(o) + (sys.getsizeof(o.__dict__) if hasattr(o, '__dict__') else 0)
             for o in series]
    unique = set(sizes)
    print(f"  {label:<16}  total sizes = {sizes}  -> all equal: {len(unique) == 1}")

# ── 2. dir() — show __dict__ / __weakref__ presence ──────────────────────────

print()
print(SEP)
print("2. dir() — presence of '__dict__' and '__weakref__'")
print(SEP)

for label, obj in (("NoSlots", n), ("WithSlots", w), ("WithSlotsWeak", ww)):
    d = dir(obj)
    has_d  = "__dict__"    in d
    has_wr = "__weakref__" in d
    print(f"  {label:<16}  '__dict__' in dir: {str(has_d):<6}  '__weakref__' in dir: {has_wr}")

# ── 3. Dynamic attribute assignment ──────────────────────────────────────────

print()
print(SEP)
print("3. Dynamic attribute assignment (obj.z = 99)")
print(SEP)

for label, obj in (("NoSlots", n), ("WithSlots", w), ("WithSlotsWeak", ww)):
    try:
        obj.z = 99
        print(f"  {label:<16}  OK  — obj.z = {obj.z}")
    except AttributeError as exc:
        print(f"  {label:<16}  AttributeError: {exc}")

# ── 4. Weak references ────────────────────────────────────────────────────────

print()
print(SEP)
print("4. Weak references (weakref.ref)")
print(SEP)

for label, obj in (("NoSlots", n), ("WithSlots", w), ("WithSlotsWeak", ww)):
    try:
        ref = weakref.ref(obj)
        print(f"  {label:<16}  OK  — weakref.ref(obj) = {ref}  ->  ref() is obj: {ref() is obj}")
    except TypeError as exc:
        print(f"  {label:<16}  TypeError: {exc}")

# ── 5. Lifecycle of a weak reference ─────────────────────────────────────────

print()
print(SEP)
print("5. Weak-reference lifecycle: what happens after the referent is deleted")
print(SEP)

# NoSlots supports weakref
tmp = NoSlots(10, 20)
wr  = weakref.ref(tmp)
print(f"  Before del tmp : wr() = {wr()}")
del tmp
print(f"  After  del tmp : wr() = {wr()}   <- referent was collected, ref returns None")

# WithSlotsWeak also supports weakref
tmp2 = WithSlotsWeak(42)
wr2  = weakref.ref(tmp2)
print(f"  Before del tmp2: wr2() = {wr2()}")
del tmp2
print(f"  After  del tmp2: wr2() = {wr2()}   <- same behaviour")

────────────────────────────────────────────────────────────
1. Structural comparison (hasattr / __slots__ / sizes)
────────────────────────────────────────────────────────────
Metric                      NoSlots    WithSlots  WithSlotsWeak
---------------------------------------------------------------
  has __dict__                 True        False          False
  has __weakref__              True        False           True
  __slots__                    None   ('x', 'y') ('x', '__weakref__')
  size_obj (B)                   48           48             56
  size_dict (B)                 264         None           None
  total_size (B)                312           48             56
  dict_keys              ['x', 'y']         None           None

  Size consistency across all 5 instances in each series:
  NoSlots           total sizes = [312, 312, 312, 312, 312]  -> all equal: True
  WithSlots         total sizes = [48, 48, 48, 48, 48]  -> all equal: True
  WithSlotsWeak     total s

## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [26]:
import sys
import weakref
import gc

SEP = '─' * 60

# ---------------------------------------------------------------------------
# Node class
# ---------------------------------------------------------------------------

class Node:
    """
    A graph node that can hold a reference to its 'parent' either as a
    strong (ordinary) reference or as a weakref.ref.

    Parameters
    ----------
    name        : human-readable label for the node
    parent      : another Node to link to (or None)
    weak_parent : if True, store the parent link as weakref.ref;
                  if False (default), store it as a plain strong reference
    """

    def __init__(self, name: str, parent=None, weak_parent: bool = False):
        self.name = name
        self._weak_parent = weak_parent

        if parent is None:
            self._parent = None
        elif weak_parent:
            # Store a weakref.ref — does NOT increment the referent's refcount
            self._parent = weakref.ref(parent)
        else:
            # Store a strong reference — increments the referent's refcount
            self._parent = parent

    def set_parent(self, parent, weak_parent: bool = False):
        """Assign (or reassign) the parent link after construction."""
        self._weak_parent = weak_parent
        if parent is None:
            self._parent = None
        elif weak_parent:
            self._parent = weakref.ref(parent)
        else:
            self._parent = parent

    def get_parent(self):
        """
        Return the parent Node (or None if the referent was collected).

        For a weak link, calling the stored weakref.ref() dereferences it:
        - returns the live object if it still exists
        - returns None if the referent has been garbage-collected
        """
        if self._parent is None:
            return None
        if self._weak_parent:
            return self._parent()   # dereference the weakref
        return self._parent

    def __repr__(self):
        link_type = 'weak' if self._weak_parent else 'strong'
        if self._parent is None:
            parent_name = None
        elif self._weak_parent:
            alive = self._parent()
            parent_name = alive.name if alive is not None else '<collected>'
        else:
            parent_name = self._parent.name
        return f'Node({self.name!r}, parent={parent_name!r}, link={link_type})'



# ===========================================================================
# PART 1 — Strong-reference cycle
# ===========================================================================
# Graph:  a.parent -> b   (strong)
#         b.parent -> a   (strong)
# Both nodes reference each other, forming a cycle.
# Even after 'del a, b' the refcounts stay > 0, so CPython's reference
# counter alone cannot free them — the cyclic GC is needed.
# ===========================================================================

print(SEP)
print('PART 1 — Strong-reference cycle')
print(SEP)

gc.disable()   # disable automatic GC so we can observe manually
gc.collect()   # flush any pre-existing garbage

a = Node('a')
b = Node('b')

# Build the cycle: a -> b -> a
a.set_parent(b, weak_parent=False)   # a._parent = b  (strong)
b.set_parent(a, weak_parent=False)   # b._parent = a  (strong)

print(f'  a = {a}')
print(f'  b = {b}')
print()

# Refcounts BEFORE deletion of external names
# Expected for 'a': 1 (name 'a') + 1 (b._parent) = 2
# Expected for 'b': 1 (name 'b') + 1 (a._parent) = 2
# sys.getrefcount() adds 1 extra ref (its own argument slot), so we subtract 1
rc_a_before = sys.getrefcount(a) - 1
rc_b_before = sys.getrefcount(b) - 1
print(f'  sys.getrefcount(a) - 1 before del: {rc_a_before}  (name "a" + b._parent)')
print(f'  sys.getrefcount(b) - 1 before del: {rc_b_before}  (name "b" + a._parent)')

print(f'\n  Strong cycle refcounts: a={rc_a_before}, b={rc_b_before}')

# Collect GC stats BEFORE deletion
unreachable_before = gc.collect()
print(f'  gc.collect() before del: {unreachable_before} unreachable objects')

# Delete the only external references
del a, b

# After 'del a, b':
#   refcount(a) drops to 1 (only b._parent still holds it)
#   refcount(b) drops to 1 (only a._parent still holds it)
# Neither reaches 0 -> CPython's refcount mechanism CANNOT free them.
# The cyclic GC must step in.
print('\n  After "del a, b":')
print('    refcounts are now 1 each (cycle keeps them alive)')
print('    CPython refcount alone cannot free them — cyclic GC needed')

unreachable_after = gc.collect()   # cyclic GC finds and frees the cycle
print(f'  gc.collect() after del:  {unreachable_after} unreachable objects collected')
print('  (both nodes freed by the cyclic garbage collector)')

gc.enable()   # re-enable automatic GC


# ===========================================================================
# PART 2 — Weak-reference cycle
# ===========================================================================
# Graph:  c.parent -> d   (strong)
#         d.parent -> c   (WEAK)
# The weak link does NOT increment c's refcount, so when 'del c' is executed
# c's refcount drops to 0 immediately — no GC cycle needed.
# ===========================================================================

print()
print(SEP)
print('PART 2 — Weak-reference cycle')
print(SEP)

gc.disable()
gc.collect()

c = Node('c')
d = Node('d')

# c -> d  (strong): d's refcount increases
c.set_parent(d, weak_parent=False)
# d -> c  (weak):   c's refcount does NOT increase
d.set_parent(c, weak_parent=True)

print(f'  c = {c}')
print(f'  d = {d}')
print()

# Refcounts:
#   c: 1 (name 'c') + 0 (d._parent is a weakref, not a strong ref) = 1
#   d: 1 (name 'd') + 1 (c._parent strong ref)                     = 2
# sys.getrefcount() adds 1 extra ref (its own argument slot), so we subtract 1
rc_c = sys.getrefcount(c) - 1
rc_d = sys.getrefcount(d) - 1
print(f'  sys.getrefcount(c) - 1: {rc_c}  (only name "c"; d._parent is a weakref — does NOT count)')
print(f'  sys.getrefcount(d) - 1: {rc_d}  (name "d" + c._parent strong ref)')
print(f'\n  Weak cycle refcounts: c={rc_c}, d={rc_d}')

# Create an external weak reference to c so we can observe its death
wr_c = weakref.ref(c)
print(f'\n  wr_c before del c: {wr_c()}  (alive)')

# Delete the external name 'c'
# c's refcount: 1 (name 'c') -> 0  => freed IMMEDIATELY by refcount mechanism
# No GC cycle needed!
del c

# wr_c() must now return None — the referent was freed
print(f'  wr_c after  del c: {wr_c()}  <- referent freed immediately (refcount -> 0)')

# d._parent (the weakref to c) also returns None now
print(f'  d.get_parent() after del c: {d.get_parent()}  <- weakref inside d also dead')

unreachable = gc.collect()
print(f'\n  gc.collect() after del c: {unreachable} unreachable objects')
print('  (c was freed by refcount, not by GC — no cycle to collect)')

gc.enable()

────────────────────────────────────────────────────────────
PART 1 — Strong-reference cycle
────────────────────────────────────────────────────────────
  a = Node('a', parent='b', link=strong)
  b = Node('b', parent='a', link=strong)

  sys.getrefcount(a) - 1 before del: 2  (name "a" + b._parent)
  sys.getrefcount(b) - 1 before del: 2  (name "b" + a._parent)

  Strong cycle refcounts: a=2, b=2
  gc.collect() before del: 0 unreachable objects

  After "del a, b":
    refcounts are now 1 each (cycle keeps them alive)
    CPython refcount alone cannot free them — cyclic GC needed
  gc.collect() after del:  2 unreachable objects collected
  (both nodes freed by the cyclic garbage collector)

────────────────────────────────────────────────────────────
PART 2 — Weak-reference cycle
────────────────────────────────────────────────────────────
  c = Node('c', parent='d', link=strong)
  d = Node('d', parent='c', link=weak)

  sys.getrefcount(c) - 1: 1  (only name "c"; d._parent is a weakref 